In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Notebook cwd:", Path.cwd().resolve())
print("Project root added:", PROJECT_ROOT)


Notebook cwd: C:\Users\yossi\SR-validation-engine\notebooks
Project root added: C:\Users\yossi\SR-validation-engine


In [2]:
import pandas as pd

from modules.data_integrity import run_integrity_checks
from modules.stress_tester import run_stress_tests, dcf_model
from modules.ai_auditor import evaluate_claims
from modules.ledger_writer import build_master_results, flatten_results, write_ledger


In [3]:
def make_integrity_input() -> tuple[pd.DataFrame, list[str]]:
    df = pd.DataFrame(
        {
            "revenue": [100.0, 200.0, None, 400.0, 200.0],
            "year": [2020, 2021, 2022, 2023, 2021],
            "ticker": ["AAPL", "AAPL", "AAPL", "AAPL", "AAPL"],
        }
    )
    return df, ["revenue", "year"]


def make_stress_inputs() -> tuple[dict[str, float], dict[str, float]]:
    base_assumptions = {
        "free_cash_flow": 500_000.0,
        "growth_rate": 0.05,
        "discount_rate": 0.10,
        "terminal_growth_rate": 0.02,
    }
    std_devs = {
        "free_cash_flow": 50_000.0,
        "growth_rate": 0.01,
        "discount_rate": 0.02,
        "terminal_growth_rate": 0.005,
    }
    return base_assumptions, std_devs


def make_audit_inputs() -> tuple[list[dict], dict]:
    claims = [
        {
            "claim_text": "Revenue was $500,000 in base year.",
            "source_key": "revenue_base",
            "extracted_value": 500_000.0,
            "unit": "usd",
            "confidence": 1.0,
        },
        {
            "claim_text": "Discount rate is approximately 12%.",
            "source_key": "discount_rate",
            "extracted_value": 0.12,
            "unit": "pct",
            "confidence": 0.8,
        },
        {
            "claim_text": "The company has strong brand moats.",
            "source_key": None,
            "extracted_value": None,
            "unit": None,
            "confidence": None,
        },
    ]
    source_truth = {
        "revenue_base": 500_000.0,
        "discount_rate": 0.10,
    }
    return claims, source_truth


integrity_df, numeric_cols = make_integrity_input()
data_integrity_results = run_integrity_checks(
    df=integrity_df,
    numeric_cols=numeric_cols,
)

base_assumptions, std_devs = make_stress_inputs()
stress_test_results = run_stress_tests(
    model_fn=dcf_model,
    base_assumptions=base_assumptions,
    std_devs=std_devs,
)

claims, source_truth = make_audit_inputs()
ai_audit_results = evaluate_claims(
    claims=claims,
    source_truth=source_truth,
)

master_results = build_master_results(
    data_integrity_results=data_integrity_results,
    stress_test_results=stress_test_results,
    ai_audit_results=ai_audit_results,
)

flat_results = flatten_results(master_results)

print("Flat result keys:")
for key in flat_results:
    print(f"  {key}")

for key, df in flat_results.items():
    assert isinstance(df, pd.DataFrame), f"{key} is not a DataFrame"

VALID_STATUSES = {"PASS", "FAIL", "N/A"}
for key, df in flat_results.items():
    if "status" not in df.columns:
        continue
    actual = set(df["status"].dropna().unique())
    assert actual.issubset(VALID_STATUSES), (
        f"{key} has unexpected status values: {actual - VALID_STATUSES}"
    )

print("\nAll assertions passed.")
print(f"Total flat result sheets: {len(flat_results)}")

output_path = write_ledger(flat_results, "SR_11_7_Validation_Ledger.xlsx")
print(f"Validation Ledger successfully written to: {output_path}")


Flat result keys:
  Data_Integrity__missing
  Data_Integrity__outliers
  Data_Integrity__duplicates
  Data_Integrity__dtypes
  Stress_Testing__sensitivity_sweep
  Stress_Testing__one_sd_summary
  AI_Audit__claim_audit
  AI_Audit__claim_summary

All assertions passed.
Total flat result sheets: 8
Validation Ledger successfully written to: SR_11_7_Validation_Ledger.xlsx
